<a href="https://colab.research.google.com/github/julia-iza/Trabajo-Practico-Laboratorio/blob/main/exploracion/inspeccion_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inspección de los datos originales — TP Subte

Este notebook **no procesa ni modifica nada**: solo lee los zips y muestra cómo viene cada año (separador, columnas, formato de las fechas y de la hora, nombres de las líneas y filas raras). Con esa información se escribe `limpieza_datos.pynb`.



## 1. Configuración

`ANIOS` está ordenado a propósito: primero los cinco años ya verificados (sirven para confirmar que la inspección detecta los problemas conocidos), después los nuevos y 2015 al final porque es el más pesado.

`LINEAS_MAX` limita cuántas filas se leen por archivo. Con `None` se lee todo. Para una prueba rápida se puede poner, por ejemplo, `200_000`, pero entonces no se ven los cambios de formato que aparecen a mitad del año.

In [ ]:
import os
import io
import time
import zipfile
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

CARPETA_ZIPS = "/content/drive/MyDrive/TP-Subte/zips"
ARCHIVO_REPORTE = "/content/drive/MyDrive/TP-Subte/inspeccion_resultado.txt"

ANIOS = [2019, 2020, 2021, 2025, 2026,   # ya verificados
         2022, 2023, 2024,               # nuevos, posteriores a la pandemia
         2016, 2017, 2018,               # nuevos, previos
         2015]                           # el más pesado, al final
LINEAS_MAX = None

for anio in ANIOS:
    ruta = os.path.join(CARPETA_ZIPS, f"molinetes-{anio}.zip")
    estado = f"{os.path.getsize(ruta) / 1_000_000:.0f} MB" if os.path.exists(ruta) else "NO ENCONTRADO"
    print(f"molinetes-{anio}.zip: {estado}")

Mounted at /content/drive
molinetes-2019.zip: 126 MB
molinetes-2020.zip: 37 MB
molinetes-2021.zip: 52 MB
molinetes-2025.zip: 82 MB
molinetes-2026.zip: 60 MB
molinetes-2022.zip: 83 MB
molinetes-2023.zip: 83 MB
molinetes-2024.zip: 79 MB
molinetes-2016.zip: 81 MB
molinetes-2017.zip: 122 MB
molinetes-2018.zip: 122 MB
molinetes-2015.zip: 922 MB


## 2. Funciones de apoyo

- `mostrar` imprime y a la vez guarda cada línea en el reporte.
- `leer_lineas` es la misma lectura que va a usar la limpieza: tolera el BOM, líneas en latin-1 dentro de archivos UTF-8 y líneas completas entre comillas. Además cuenta cuántas veces pasó cada cosa.
- `forma` reemplaza los dígitos por `9`, para ver el "molde" de una fecha: `01/02/2020` es `99/99/9999` y `3/13/2020` es `9/99/9999`.
- `clasificar_fecha` dice si una fecha se puede leer de una sola manera o es ambigua. Por ejemplo, en `13/3/2020` el 13 no puede ser un mes, así que seguro es día/mes; en `3/4/2020` no hay forma de saberlo mirando solo ese texto.

In [ ]:
reporte = []

def mostrar(texto=""):
    """Imprime el texto y lo guarda para el reporte final."""
    print(texto)
    reporte.append(str(texto))

def guardar_reporte():
    with open(ARCHIVO_REPORTE, "w", encoding="utf-8") as f:
        f.write("\n".join(reporte) + "\n")

def leer_lineas(archivo_abierto, contadores):
    """Lee línea por línea: saca el BOM, tolera latin-1 y quita comillas envolventes."""
    primera = True
    for crudo in archivo_abierto:
        try:
            linea = crudo.decode("utf-8-sig" if primera else "utf-8")
        except UnicodeDecodeError:
            linea = crudo.decode("latin-1")
            contadores["latin1"] += 1
        primera = False
        linea = linea.strip()
        if linea.startswith('"') and linea.rfind('"') > 0:
            contadores["entre_comillas"] += 1
            if linea.rfind('"') < len(linea) - 1:
                contadores["texto_despues_de_comillas"] += 1   # por ejemplo ';;;;' sobrantes
            linea = linea[1:linea.rfind('"')]
        yield linea

def forma(texto):
    """Reemplaza cada dígito por 9 para ver el formato: '3/13/2020' -> '9/99/9999'."""
    resultado = ""
    for caracter in texto:
        if caracter.isdigit():
            resultado += "9"
        else:
            resultado += caracter
    return resultado

def clasificar_fecha(texto):
    """Indica si la fecha tiene una sola lectura posible o es ambigua."""
    try:
        solo_fecha = texto.split(" ")[0]          # por si viene con hora: '2015-01-01 00:00:00'
        if "/" in solo_fecha:
            a, b = [int(x) for x in solo_fecha.split("/")[:2]]
            if a > 12:
                return "barras: seguro dia/mes"
            if b > 12:
                return "barras: seguro mes/dia"
            return "barras: ambigua"
        if "-" in solo_fecha:
            partes = [int(x) for x in solo_fecha.split("-")[:3]]
            if partes[1] > 12:
                return "guiones: seguro AAAA-DD-MM (invertida)"
            if partes[2] > 12:
                return "guiones: seguro AAAA-MM-DD (normal)"
            return "guiones: ambigua"
    except (ValueError, IndexError):
        pass
    return "otro formato"

def sumar(diccionario, clave):
    diccionario[clave] = diccionario.get(clave, 0) + 1

def ordenar(diccionario, cuantos=10):
    """Las claves más frecuentes, de mayor a menor."""
    pares = sorted(diccionario.items(), key=lambda par: par[1], reverse=True)
    return pares[:cuantos]

print("Funciones listas.")

Funciones listas.


## 3. Inspección de un archivo CSV

Para cada CSV dentro de un zip:

1. Muestra el encabezado tal cual viene y detecta el separador (`,` o `;`).
2. Muestra las primeras filas.
3. Recorre todas las filas y anota:
   - cuántas filas tiene y cuántos separadores trae cada una (si varía, hay filas rotas o columnas de más);
   - el molde de las fechas y si son ambiguas, una vez por cada fecha distinta;
   - una muestra de fechas repartida a lo largo del archivo, para ver si el formato cambia a mitad de año;
   - cuántas veces la fecha "vuelve para atrás", lo que pasa cuando el archivo reinicia un bloque de líneas;
   - el largo del texto de la hora (`12:15` contra `12:15:00`);
   - los nombres de las líneas de subte y cuántas filas tiene cada una.

In [ ]:
def inspeccionar_csv(archivo_abierto, nombre, anio):
    contadores = {"latin1": 0, "entre_comillas": 0, "texto_despues_de_comillas": 0}
    lineas = leer_lineas(archivo_abierto, contadores)

    try:
        encabezado = next(lineas)
    except StopIteration:
        mostrar("  (archivo vacío)")
        return None

    sep = ";" if encabezado.count(";") > encabezado.count(",") else ","
    columnas = [c.lower().strip().strip('"') for c in encabezado.split(sep)]
    mostrar(f"  Encabezado crudo: {encabezado[:250]!r}")
    mostrar(f"  Separador: {sep!r} | {len(columnas)} columnas: {columnas}")

    pos_fecha = columnas.index("fecha") if "fecha" in columnas else None
    pos_desde = columnas.index("desde") if "desde" in columnas else None
    pos_linea = columnas.index("linea") if "linea" in columnas else None
    if pos_fecha is None or pos_linea is None:
        mostrar("  ATENCIÓN: no hay columna 'fecha' o 'linea' con ese nombre; solo se cuentan filas.")
    posiciones = [p for p in [pos_fecha, pos_desde, pos_linea] if p is not None]
    ultima_pos = max(posiciones) if posiciones else 0

    filas = 0
    ejemplos = []
    separadores_por_fila = {}
    formas_fecha = {}
    tipos_fecha = {}
    ejemplo_por_tipo = {}
    largos_hora = {}
    ejemplo_por_largo = {}
    lineas_subte = {}
    secuencia_fechas = []        # cada fecha distinta, en el orden en que aparece
    filas_sin_fecha = 0
    filas_cortas = 0
    ultima_fecha = None
    inicio = time.time()

    for linea in lineas:
        filas += 1
        if len(ejemplos) < 3:
            ejemplos.append(linea[:250])
        sumar(separadores_por_fila, linea.count(sep))

        if posiciones:
            partes = linea.split(sep, ultima_pos + 1)
            if len(partes) <= ultima_pos:
                filas_cortas += 1
                continue

            if pos_fecha is not None:
                texto = partes[pos_fecha].strip().strip('"')
                if texto == "":
                    filas_sin_fecha += 1
                elif texto != ultima_fecha:          # solo se analiza cuando la fecha cambia
                    ultima_fecha = texto
                    secuencia_fechas.append(texto)
                    sumar(formas_fecha, forma(texto))
                    tipo = clasificar_fecha(texto)
                    sumar(tipos_fecha, tipo)
                    ejemplo_por_tipo.setdefault(tipo, texto)

            if pos_desde is not None:
                hora = partes[pos_desde].strip().strip('"')
                sumar(largos_hora, len(hora))
                ejemplo_por_largo.setdefault(len(hora), hora)

            if pos_linea is not None:
                sumar(lineas_subte, partes[pos_linea].strip().strip('"'))

        if LINEAS_MAX is not None and filas >= LINEAS_MAX:
            mostrar(f"  (cortado en {LINEAS_MAX:,} filas por LINEAS_MAX)")
            break

    mostrar("  Primeras filas:")
    for ejemplo in ejemplos:
        mostrar(f"    {ejemplo!r}")
    mostrar(f"  Filas de datos: {filas:,} | leídas en {time.time() - inicio:.0f} s")
    mostrar(f"  Separadores por fila (cantidad: filas): {ordenar(separadores_por_fila, 5)}")
    mostrar(f"  Líneas en latin-1: {contadores['latin1']} | entre comillas: {contadores['entre_comillas']:,}"
            f" | con texto después de las comillas: {contadores['texto_despues_de_comillas']:,}")
    if filas_cortas:
        mostrar(f"  Filas con menos columnas de las esperadas: {filas_cortas:,}")

    if pos_fecha is not None:
        mostrar(f"  Filas sin fecha: {filas_sin_fecha:,} | cambios de fecha: {len(secuencia_fechas):,}"
                f" | textos de fecha distintos: {len(set(secuencia_fechas)):,}")
        mostrar(f"  Moldes de fecha: {ordenar(formas_fecha)}")
        for tipo, cantidad in ordenar(tipos_fecha):
            mostrar(f"    {tipo}: {cantidad:,} (ej. {ejemplo_por_tipo[tipo]!r})")
        if secuencia_fechas:
            paso = max(1, len(secuencia_fechas) // 12)
            muestra = secuencia_fechas[::paso] + [secuencia_fechas[-1]]
            mostrar(f"  Fechas a lo largo del archivo: {muestra}")

    if pos_desde is not None:
        largos = [(largo, cantidad, ejemplo_por_largo[largo]) for largo, cantidad in ordenar(largos_hora, 5)]
        mostrar(f"  Hora 'desde' (largo, filas, ejemplo): {largos}")

    if pos_linea is not None:
        mostrar(f"  Líneas de subte (nombre: filas): {ordenar(lineas_subte, 15)}")

    return {
        "anio": anio, "archivo": nombre, "separador": sep, "columnas": len(columnas),
        "filas": filas, "filas_sin_fecha": filas_sin_fecha,
        "fechas_ambiguas": sum(c for t, c in tipos_fecha.items() if "ambigua" in t),
        "moldes_fecha": " | ".join(formas_fecha.keys()),
        "latin1": contadores["latin1"],
        "lineas_subte": len(lineas_subte),
    }

print("Función de inspección lista.")

Función de inspección lista.


## 4. Inspección de un zip

Lista todo lo que hay adentro del zip con su tamaño descomprimido e inspecciona cada CSV. Si adentro hay otro zip (pasa en algunos datasets), también lo abre.

In [ ]:
def inspeccionar_zip(zf, anio, prefijo=""):
    resultados = []
    contenido = [i for i in zf.infolist() if not i.is_dir()]
    mostrar(f"{prefijo}Archivos adentro: {len(contenido)}")
    for info in contenido:
        mostrar(f"{prefijo}  - {info.filename} ({info.file_size / 1_000_000:.1f} MB descomprimido)")

    for info in contenido:
        nombre = info.filename
        if nombre.lower().endswith(".zip"):
            mostrar(f"\n{prefijo}>> Zip dentro del zip: {nombre}")
            with zf.open(nombre) as interno:
                with zipfile.ZipFile(io.BytesIO(interno.read())) as zf_interno:
                    resultados += inspeccionar_zip(zf_interno, anio, prefijo + "  ")
        elif nombre.lower().endswith((".csv", ".txt")):
            mostrar(f"\n{prefijo}>> {nombre}")
            with zf.open(nombre) as archivo:
                resultado = inspeccionar_csv(archivo, nombre, anio)
            if resultado:
                resultados.append(resultado)
        else:
            mostrar(f"\n{prefijo}>> {nombre}: no es CSV, se saltea.")
    return resultados

print("Función de zips lista.")

Función de zips lista.


## 5. Correr la inspección para todos los años

Esta es la celda que tarda. Va mostrando cada archivo a medida que lo termina y guarda el reporte en Drive después de cada año.

In [ ]:
reporte = []
resumen = []
inicio_total = time.time()

for anio in ANIOS:
    ruta = os.path.join(CARPETA_ZIPS, f"molinetes-{anio}.zip")
    mostrar("\n" + "=" * 90)
    mostrar(f"AÑO {anio}  —  molinetes-{anio}.zip")
    mostrar("=" * 90)
    if not os.path.exists(ruta):
        mostrar("NO ENCONTRADO en Drive, se saltea.")
        continue
    inicio = time.time()
    try:
        with zipfile.ZipFile(ruta) as zf:
            resumen += inspeccionar_zip(zf, anio)
    except Exception as error:
        mostrar(f"ERROR al leer {anio}: {type(error).__name__}: {error}")
    mostrar(f"\nAño {anio} terminado en {(time.time() - inicio) / 60:.1f} min.")
    guardar_reporte()

mostrar(f"\nInspección completa en {(time.time() - inicio_total) / 60:.1f} min.")
guardar_reporte()


AÑO 2019  —  molinetes-2019.zip
Archivos adentro: 1
  - datahistorica122019.csv (1238.7 MB descomprimido)

>> datahistorica122019.csv
  Encabezado crudo: 'periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total'
  Separador: ',' | 11 columnas: ['periodo', 'fecha', 'desde', 'hasta', 'linea', 'molinete', 'estacion', 'pax_pagos', 'pax_pases_pagos', 'pax_franq', 'total']
  Primeras filas:
    '201901,2019-01-01,08:00:00,08:15:00,LineaA,LineaA_Lima_N_Turn02,Lima,1.0,0.0,0.0,1.0'
    '201901,2019-01-01,08:00:00,08:15:00,LineaA,LineaA_Loria_N_Turn03,Loria,3.0,0.0,0.0,3.0'
    '201901,2019-01-01,08:00:00,08:15:00,LineaA,LineaA_Miserere_Q_HALL_Turn01,Plaza Miserere,3.0,0.0,0.0,3.0'
  Filas de datos: 12,662,343 | leídas en 52 s
  Separadores por fila (cantidad: filas): [(10, 12662343)]
  Líneas en latin-1: 0 | entre comillas: 0 | con texto después de las comillas: 0
  Filas sin fecha: 0 | cambios de fecha: 423 | textos de fecha distintos: 362
  Moldes de fech

## 6. Resumen

Una fila por cada CSV. Sirve para comparar los años de un vistazo: si un año tiene otro separador, otra cantidad de columnas, otros moldes de fecha o muchas fechas ambiguas, hay que tratarlo distinto en la limpieza.

In [ ]:
tabla = pd.DataFrame(resumen)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60)

mostrar("\n" + "=" * 90)
mostrar("RESUMEN POR ARCHIVO")
mostrar("=" * 90)
mostrar(tabla.to_string(index=False))

mostrar("\nRESUMEN POR AÑO")
por_anio = tabla.groupby("anio").agg(archivos=("archivo", "count"),
                                     filas=("filas", "sum"),
                                     filas_sin_fecha=("filas_sin_fecha", "sum"),
                                     fechas_ambiguas=("fechas_ambiguas", "sum"),
                                     latin1=("latin1", "sum"))
mostrar(por_anio.to_string())
guardar_reporte()
print(f"\nReporte guardado en: {ARCHIVO_REPORTE}")
tabla


RESUMEN POR ARCHIVO
 anio                                                           archivo separador  columnas    filas  filas_sin_fecha  fechas_ambiguas                                               moldes_fecha  latin1  lineas_subte
 2019                                           datahistorica122019.csv         ,        11 12662343                0              167                                                 9999-99-99       0             6
 2020                                                    historico2.csv         ,        10  5781006             1836              192              99/99/9999 | 9/9/9999 | 9/99/9999 | 99/9/9999       0             7
 2021                                                historico_2021.csv         ;        11  8071680                0              204 9/9/9999 | 99/9/9999 | 9999-99-99 | 9/99/9999 | 99/99/9999       0             6
 2025     molinetes-2025/202501_PAX15min-ABC-INCLUYEOTROMODOSDEPAGO.csv         ;        10   517760               

,anio,archivo,separador,columnas,filas,filas_sin_fecha,fechas_ambiguas,moldes_fecha,latin1,lineas_subte
0,2019,datahistorica122019.csv,",",11,12662343,0,167,9999-99-99,0,6
1,2020,historico2.csv,",",10,5781006,1836,192,99/99/9999 | 9/9/9999 | 9/99/9999 | 99/9/9999,0,7
2,2021,historico_2021.csv,;,11,8071680,0,204,9/9/9999 | 99/9/9999 | 9999-99-99 | 9/99/9999 | 99/99/9999,0,6
3,2025,molinetes-2025/202501_PAX15min-ABC-INCLUYEOTROMODOSDEPAG...,;,10,517760,0,527,9/9/9999 | 99/9/9999,0,3
4,2025,molinetes-2025/202501_PAX15min-DEH-INCLUYEOTROMODOSDEPAG...,;,10,492193,0,539,9/9/9999 | 99/9/9999,0,3
...,...,...,...,...,...,...,...,...,...,...
224,2015,molinetes-2025/202510_PAX15min-DEH-INCLUYEOTROMODOSDEPAG...,;,10,470664,0,24,9/99/9999 | 99/99/9999,0,3
225,2015,molinetes-2025/202511_PAX15min-ABC-INCLUYEOTROMODOSDEPAG...,;,10,536844,0,24,9/99/9999 | 99/99/9999,0,3
226,2015,molinetes-2025/202511_PAX15min-DEH-INCLUYEOTROMODOSDEPAG...,;,10,443958,0,24,9/99/9999 | 99/99/9999,0,3
227,2015,molinetes-2025/202512_PAX15min-ABC-INCLUYEOTROMODOSDEPAG...,;,10,555937,0,60,9/99/9999 | 99/99/9999,10397,3


## 7. Qué hacer con el resultado

El reporte quedó en `Mi unidad/TP-Subte/inspeccion_resultado.txt`. Descargalo desde Drive y adjuntalo en el chat con Claude: con eso se ajusta `src/limpieza.py` a los formatos de cada año.